In [40]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 42, Finished, Available, Finished, False)

In [45]:
#cleaning customer
customer = spark.table("Bronze.dbo.customers")

customer_clean = (
    customer
    .withColumn("email", lower(trim(col("EMAIL"))))
    .withColumn("name", initcap(trim(col("name"))))
    .withColumn("gender", when(lower(col("gender")).isin("f", "female"), "Female")
    .when(lower(col("gender")).isin("m", "male"), "Male")
    .otherwise("Other"))
    .withColumn("location",initcap(col("location")))
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id","email"])
    .withColumn(
    "dob",
    coalesce(
        to_date(col("dob"), "yyyy/MM/dd"),
        to_date(col("dob"), "dd-MM-yyyy"),
        to_date(col("dob"), "yyyy-MM-dd"),
        to_date(col("dob"), "dd/MM/yyyy")
    )
)
)
customer_clean = customer_clean.dropna(subset=["dob"])
display(customer_clean)
customer_clean.write.format("delta").mode("overwrite").saveAsTable("EcommerceWorkspace.Silver.dbo.customers")

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 47, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 11604e5c-108b-446e-ad11-1dd2c6290cbf)

In [20]:
#Cleaning Order
orders = spark.table("dbo.orders")
display(orders)

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 22, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a894c5f1-5f95-421a-8039-d083aa7c9d18)

In [46]:
spark.sql("DROP TABLE Silver.dbo.orders")

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 48, Finished, Available, Finished, False)

DataFrame[]

In [47]:
orders_clean = (
    orders
    .withColumn("amount", when(col("amount")<0,abs("amount")).otherwise(col("amount")))
    .withColumn("amount", when(col("amount").isNull(), 0).otherwise(col("amount")))
    .withColumn("status", initcap(col("status")))
    .dropna(subset=["customer_id","order_date"])
    .dropDuplicates(["order_id"])
    .withColumn("order_date", coalesce(to_date(col("order_date"), "yyyy/MM/dd"),
    to_date(col("order_date"), "yyyy-MM-dd"),
    to_date(col("order_date"), "yyyyMMdd"),
    to_date(col("order_date"), "dd/MM/yyyy")))

)
display(orders_clean)
orders_clean.write.format("delta").mode("overwrite").saveAsTable("Silver.dbo.orders")

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 49, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a3581918-2dd8-4094-95d0-6d5d308e6788)

In [48]:
spark.sql("DROP TABLE Silver.dbo.payments")

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 50, Finished, Available, Finished, False)

DataFrame[]

In [49]:
payments = spark.table("dbo.payments")
display(payments)

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 51, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 69b1c58d-80cd-4035-aadc-533ed5d1149a)

In [55]:
#cleaning payments
payments = spark.table("dbo.payments")
payments_clean = (
    payments.withColumn("payment_method", initcap(trim(col("payment_method"))))
    .withColumn("payment_date",trim(col("payment_date")))
    .withColumn("amount",when(trim(col("amount")).isNull(),0).otherwise(abs(col("amount"))))
    .withColumn("payment_status",when(trim(col("payment_status")).isNull(),"NA").otherwise(col("payment_status")))
    .replace({"Creditcard":"Credit Card","Paypal":"PayPal","n/a":"NA","NULL":"NA"}, subset=["payment_method","payment_date","payment_status"])
    .replace({"NULL":"0"}, subset=["amount"])
    .withColumn("payment_status", initcap(col("payment_status")))
    .withColumn("amount", when(col("amount")<0, abs(col("amount"))).otherwise(col("amount")))
    .dropna(subset=["customer_id","payment_date"])
    .withColumn("payment_date", coalesce( 
    to_date(col("payment_date"), "yyyy/MM/dd"),
    to_date(col("payment_date"), "yyyy-MM-dd"),
    to_date(col("payment_date"), "yyyyMMdd"),
    to_date(col("payment_date"), "dd/MM/yyyy")))
)
display(payments_clean)
payments_clean.write.format("delta").mode("overwrite").saveAsTable("Silver.dbo.payments")

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 57, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 211394e9-d40e-4d25-b732-5393ee3b28e6)

In [56]:
spark.sql("DROP TABLE Silver.dbo.support_tickets")

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 58, Finished, Available, Finished, False)

DataFrame[]

In [57]:
#cleaning support_tickets
support_tickets = spark.table("Bronze.dbo.support_tickets")
display(support_tickets)

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 59, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 9af56ddd-0ff3-4af3-8f95-250d679174c1)

In [60]:
from pyspark.sql.functions import *
from pyspark.sql.types import DoubleType
support_tickets_clean = (
    support_tickets.withColumn("issue_type", initcap(trim(col("issue_type"))))
    .withColumn("resolution_status", initcap(trim(col("resolution_status"))))
    .withColumn("resolution_status", when(col("resolution_status").isNull(),"NA").otherwise(col("resolution_status")))
    .replace({"NA": "NA","": "NA","n/a":"NA","Na":"NA"}, subset=["issue_type","resolution_status","ticket_date"])
    .dropDuplicates(["ticket_date"])
    .dropna(subset=["customer_id","ticket_date"])
    .withColumn("ticket_date", coalesce( 
    to_date(col("ticket_date"), "yyyy/MM/dd"),
    to_date(col("ticket_date"), "yyyy-MM-dd"),
    to_date(col("ticket_date"), "yyyyMMdd"),
    to_date(col("ticket_date"), "dd/MM/yyyy")))
    )

display(support_tickets_clean)

support_tickets_clean.write.format("delta").mode("overwrite").saveAsTable("Silver.dbo.support_tickets")

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 62, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, e15cab37-6eab-4555-9c7e-2905cb108f2b)

In [61]:
spark.sql("DROP TABLE Silver.dbo.web_activities")

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 63, Finished, Available, Finished, False)

DataFrame[]

In [62]:
#Cleaning web_activities
web_activities = spark.table("Bronze.dbo.web_activities")
display(web_activities)

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 64, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4f01116a-5ce0-45a6-801e-8d504f388903)

In [65]:
web_activities_clean = (
    web_activities.withColumn("device_type", initcap(trim(col("device_type"))))
    .dropDuplicates(subset=["session_id","customer_id"])
    .withColumn("page_viewed", lower(col("page_viewed")))
    .dropna(subset=["customer_id","session_time","page_viewed"])
    .withColumn("session_time", coalesce( 
    to_date(col("session_time"), "yyyy/MM/dd"),
    to_date(col("session_time"), "yyyy-MM-dd"),
    to_date(col("session_time"), "yyyyMMdd"),
    to_date(col("session_time"), "dd/MM/yyyy")))
)
display(web_activities_clean)
web_activities_clean.write.format("delta").mode("overwrite").saveAsTable("Silver.dbo.web_activities")

StatementMeta(, dce08232-6b19-45ca-bb19-08bf01543a7a, 67, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a3f165a9-a6d1-4f6d-ae50-ca04fad401cb)